# Classroom LLM gateway test

This notebook verifies that your JupyterHub identity received a unique LiteLLM virtual key and that requests reach the GPU-hosted Qwen model through the allow-listed gateway. The key is read from the environment and is never printed.


In [ ]:
import os
from openai import OpenAI

required = ["OPENAI_BASE_URL", "OPENAI_API_KEY", "CLASSROOM_MODEL"]
missing = [name for name in required if not os.getenv(name)]
assert not missing, f"Missing environment variables: {missing}"

print({
    "username": os.getenv("JUPYTERHUB_USERNAME"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
    "model": os.getenv("CLASSROOM_MODEL"),
    "api_key_present": bool(os.getenv("OPENAI_API_KEY")),
})


In [ ]:
client = OpenAI(
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    timeout=300,
)

response = client.chat.completions.create(
    model=os.environ["CLASSROOM_MODEL"],
    messages=[
        {"role": "system", "content": "You are a concise biology coding tutor."},
        {"role": "user", "content": "Write a Python function that calculates GC percentage and validates DNA characters."},
    ],
    temperature=0.2,
    max_tokens=300,
)
print(response.choices[0].message.content)
print("Usage:", response.usage)


## What this proves

- The notebook can reach the classroom gateway.
- The gateway accepts this user's virtual key.
- LiteLLM forwards the request to vLLM.
- vLLM serves the local Qwen3.5-9B checkpoint on GPU 0.
- The response includes input and output token counts.

It does **not** prove that the public HTTPS route is available. Test that separately using the same key and the instructor-provided public base URL.
